# RecursiveCharacterTextSplitter

일반 텍스트에 **가장 먼저 권장되는** 분할기입니다. (LangChain 공식 문서도 "대부분의 경우 이것부터 시작하라"고 안내합니다.)

구분자 목록을 순서대로 시도하면서, 청크가 충분히 작아질 때까지 재귀적으로 분할합니다.
기본 구분자 목록은 `["\n\n", "\n", " ", ""]` 입니다.

- **단락** → **문장(줄)** → **단어** → **글자** 순서로 분할
- 의미적으로 강하게 연결된 단위(단락)를 최대한 함께 유지하려는 효과가 있습니다.

1. 텍스트 분할 방식: 구분자 목록(`["\n\n", "\n", " ", ""]`) 에 의해 분할
2. 청크 크기 측정 방식: 문자 수

> **🔄 최신 버전 기준 변경 사항 (langchain-text-splitters 1.x)**
> - 파일 읽기 시 `encoding="utf-8"` 명시
> - 기본 사용법은 그대로이며, 한국어 문장 경계를 고려한 **사용자 정의 `separators`** 와 `add_start_index` 예시를 추가했습니다.
> - 토큰 수 기준으로 크기를 재는 `from_tiktoken_encoder()` 는 03 노트북에서 다룹니다.

In [ ]:
%pip install -qU langchain-text-splitters

In [ ]:
from pathlib import Path

file = Path("./data/appendix-keywords.txt").read_text(encoding="utf-8")
print(file[:500])

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

- `chunk_size=250`: 각 청크의 최대 크기
- `chunk_overlap=50`: 인접 청크 간 50자 중첩 허용
- `length_function=len`: 문자 수로 길이 계산
- `is_separator_regex=False`: 구분자를 정규식으로 해석하지 않음
- `separators`: 생략하면 기본값 `["\n\n", "\n", " ", ""]` 가 사용됩니다. 여기서는 동작을 명확히 보기 위해 명시했습니다.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],  # 기본값과 동일 (명시적으로 작성)
    chunk_size=250,
    chunk_overlap=50,
    length_function=len,
    is_separator_regex=False,
)

In [ ]:
texts = text_splitter.create_documents([file])
print(texts[0])
print("===" * 20)
print(texts[1])

In [ ]:
# 문자열 리스트로 받고 싶다면 split_text()를 사용합니다.
text_splitter.split_text(file)[:2]

## 🔄 추가: 한국어 문장 경계를 고려한 구분자

기본 구분자에는 문장 부호가 없어서, 줄바꿈이 없는 긴 문단은 **공백(단어) 단위**로 잘립니다.
한국어/영어 문장 끝(`. `, `? `, `! `)을 단어보다 먼저 시도하도록 구분자를 추가하면 문장 중간에서 끊기는 경우를 줄일 수 있습니다.

- `keep_separator="end"`: 구분자(마침표 등)를 앞 청크의 **끝**에 붙여 문장이 자연스럽게 끝나도록 합니다. (기본값 `True` 는 다음 청크의 시작에 붙입니다.)
- `add_start_index=True`: 원문 내 시작 위치를 메타데이터에 기록합니다.

In [ ]:
ko_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", "? ", "! ", " ", ""],
    keep_separator="end",
    chunk_size=250,
    chunk_overlap=50,
    add_start_index=True,
)

ko_docs = ko_splitter.create_documents([file])

print(f"기본 구분자 청크 수: {len(texts)} / 사용자 정의 구분자 청크 수: {len(ko_docs)}")
for doc in ko_docs[:3]:
    print(doc.metadata)
    print(doc.page_content)
    print("---" * 20)

## 🔄 추가: 이미 로드된 Document 분할하기

RAG 파이프라인에서는 보통 `Document` 리스트를 `split_documents()` 로 분할합니다. 원본 메타데이터는 모든 청크에 복사됩니다.

In [ ]:
from langchain_core.documents import Document

source_docs = [Document(page_content=file, metadata={"source": "appendix-keywords.txt"})]
splits = ko_splitter.split_documents(source_docs)
splits[0].metadata